# 02 — Build Evaluation Prompts

Questo notebook prepara il test set utilizzato per confrontare:

- il modello base `Qwen/Qwen2.5-3B-Instruct`;
- l'adapter `regulation`;
- l'adapter `positive_legends`;
- l'adapter `negative_legends`;
- l'adapter `negative_to_correction`.

Il notebook:

1. legge il test set in formato JSONL;
2. valida la struttura degli 8 task previsti;
3. esclude esplicitamente `qnli_style`;
4. normalizza gli esempi in uno schema comune;
5. costruisce i messaggi da inviare al modello;
6. applica il chat template ufficiale di Qwen;
7. salva i prompt pronti per il notebook di inferenza.

La ground truth viene mantenuta nel record per il calcolo delle metriche,
ma non viene mai inserita nel prompt inviato al modello.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import hashlib
import json
import os
import sys

from transformers import AutoTokenizer


print("Python:", sys.version)
print("Directory corrente:", Path.cwd())

C:\Users\fede2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Directory corrente: c:\Users\fede2\Desktop\Project


In [2]:
def find_project_root(start_path: Path) -> Path:
    """
    Individua la root del progetto risalendo dalla directory corrente.

    La root viene riconosciuta dalla presenza di almeno due tra:
    - data
    - models
    - notebooks
    """

    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        expected_directories = [
            candidate / "data",
            candidate / "models",
            candidate / "notebooks",
        ]

        existing_directories = sum(
            path.exists()
            for path in expected_directories
        )

        if existing_directories >= 2:
            return candidate

    raise FileNotFoundError(
        "Impossibile individuare automaticamente la root del progetto. "
        "Eseguire il notebook dalla directory del progetto oppure "
        "impostare manualmente PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\fede2\Desktop\Project


Cella 5 — Configurazione generale

Il file JSONL deve essere collocato qui:

data/raw/evaluation/test_set.jsonl

Ti consiglio di rinominare il file eliminando lo spazio da:

test set.jsonl

a:

test_set.jsonl

In [3]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OFFLINE_MODE = False

INPUT_JSONL_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "evaluation"
    / "test_set.jsonl"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "evaluation"
)

OUTPUT_BY_TASK_DIR = (
    OUTPUT_ROOT
    / "by_task"
)

OUTPUT_NORMALIZED_PATH = (
    OUTPUT_ROOT
    / "evaluation_normalized.jsonl"
)

OUTPUT_ALL_PROMPTS_PATH = (
    OUTPUT_ROOT
    / "evaluation_prompts_all.jsonl"
)

MANIFEST_DIR = (
    PROJECT_ROOT
    / "manifests"
)

MANIFEST_PATH = (
    MANIFEST_DIR
    / "02_build_evaluation_prompts_manifest.json"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_BY_TASK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Input JSONL:", INPUT_JSONL_PATH)
print("Output normalizzato:", OUTPUT_NORMALIZED_PATH)
print("Output prompt:", OUTPUT_ALL_PROMPTS_PATH)
print("Output per task:", OUTPUT_BY_TASK_DIR)
print("Manifest:", MANIFEST_PATH)

Input JSONL: C:\Users\fede2\Desktop\Project\data\raw\evaluation\test_set.jsonl
Output normalizzato: C:\Users\fede2\Desktop\Project\data\processed\evaluation\evaluation_normalized.jsonl
Output prompt: C:\Users\fede2\Desktop\Project\data\processed\evaluation\evaluation_prompts_all.jsonl
Output per task: C:\Users\fede2\Desktop\Project\data\processed\evaluation\by_task
Manifest: C:\Users\fede2\Desktop\Project\manifests\02_build_evaluation_prompts_manifest.json


In [4]:
TASK_CONFIG = {
    "sst2_style": {
        "task_type": "classification",
        "input_fields": [
            "text",
        ],
        "allowed_labels": [
            "compliant",
            "non_compliant",
        ],
        "max_new_tokens": 8,
    },

    "paraphrase_style": {
        "task_type": "classification",
        "input_fields": [
            "sentence_1",
            "sentence_2",
        ],
        "allowed_labels": [
            "equivalent",
            "not_equivalent",
        ],
        "max_new_tokens": 8,
    },

    "stsb_style": {
        "task_type": "ordinal_regression",
        "input_fields": [
            "sentence_1",
            "sentence_2",
        ],
        "allowed_labels": [
            0,
            1,
            2,
            3,
            4,
            5,
        ],
        "max_new_tokens": 4,
    },

    "qqp_style": {
        "task_type": "classification",
        "input_fields": [
            "question_1",
            "question_2",
        ],
        "allowed_labels": [
            "duplicate",
            "not_duplicate",
        ],
        "max_new_tokens": 8,
    },

    "mnli_style": {
        "task_type": "classification",
        "input_fields": [
            "premise",
            "hypothesis",
        ],
        "allowed_labels": [
            "entailment",
            "contradiction",
            "neutral",
        ],
        "max_new_tokens": 8,
    },

    "rte_style": {
        "task_type": "classification",
        "input_fields": [
            "premise",
            "hypothesis",
        ],
        "allowed_labels": [
            "entailment",
            "not_entailment",
        ],
        "max_new_tokens": 8,
    },

    "bias_detection": {
        "task_type": "classification",
        "input_fields": [
            "text",
        ],
        "allowed_labels": [
            "biased",
            "unbiased",
        ],
        "max_new_tokens": 8,
    },

    "grounded_answer_generation": {
        "task_type": "generation",
        "input_fields": [
            "text",
        ],
        "allowed_labels": None,
        "max_new_tokens": 192,
    },
}


EXPECTED_TASK_COUNTS = {
    "sst2_style": 20,
    "paraphrase_style": 20,
    "stsb_style": 20,
    "qqp_style": 20,
    "mnli_style": 20,
    "rte_style": 20,
    "bias_detection": 20,
    "grounded_answer_generation": 20,
}

EXPECTED_TOTAL_RECORDS = sum(
    EXPECTED_TASK_COUNTS.values()
)

FORBIDDEN_TASKS = {
    "qnli_style",
}

print("Task configurati:", len(TASK_CONFIG))
print("Record complessivi attesi:", EXPECTED_TOTAL_RECORDS)

for task_name, config in TASK_CONFIG.items():
    print(
        f"- {task_name}: "
        f"{config['task_type']} | "
        f"input={config['input_fields']} | "
        f"record attesi={EXPECTED_TASK_COUNTS[task_name]}"
    )

Task configurati: 8
Record complessivi attesi: 160
- sst2_style: classification | input=['text'] | record attesi=20
- paraphrase_style: classification | input=['sentence_1', 'sentence_2'] | record attesi=20
- stsb_style: ordinal_regression | input=['sentence_1', 'sentence_2'] | record attesi=20
- qqp_style: classification | input=['question_1', 'question_2'] | record attesi=20
- mnli_style: classification | input=['premise', 'hypothesis'] | record attesi=20
- rte_style: classification | input=['premise', 'hypothesis'] | record attesi=20
- bias_detection: classification | input=['text'] | record attesi=20
- grounded_answer_generation: generation | input=['text'] | record attesi=20


In [5]:
if not INPUT_JSONL_PATH.exists():
    raise FileNotFoundError(
        "Il test set JSONL non è stato trovato.\n\n"
        f"Percorso atteso:\n{INPUT_JSONL_PATH}\n\n"
        "Copiare o rinominare il file come test_set.jsonl "
        "all'interno di data/raw/evaluation/."
    )

print("JSONL trovato:", INPUT_JSONL_PATH)
print("Dimensione:", INPUT_JSONL_PATH.stat().st_size, "byte")

JSONL trovato: C:\Users\fede2\Desktop\Project\data\raw\evaluation\test_set.jsonl
Dimensione: 52142 byte


In [6]:
def read_jsonl(path: Path) -> list[dict]:
    """
    Legge un file JSONL.

    Ogni riga non vuota deve contenere un oggetto JSON.
    """

    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            stripped_line = line.strip()

            if not stripped_line:
                continue

            try:
                record = json.loads(
                    stripped_line
                )

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"JSON non valido alla riga {line_number} "
                    f"del file {path}:\n{error}"
                ) from error

            if not isinstance(record, dict):
                raise TypeError(
                    f"La riga {line_number} non contiene "
                    "un oggetto JSON."
                )

            record["_source_line"] = line_number

            records.append(record)

    return records


def write_jsonl(
    path: Path,
    records: list[dict],
) -> None:
    """
    Scrive una lista di dizionari in formato JSONL.
    """

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
            )

            file.write("\n")


def write_json(
    path: Path,
    data: dict,
) -> None:
    """
    Scrive un oggetto JSON formattato.
    """

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

In [7]:
raw_records = read_jsonl(
    INPUT_JSONL_PATH
)

print("Record caricati:", len(raw_records))

Record caricati: 160


In [8]:
raw_task_counts = Counter(
    record.get("task_name")
    for record in raw_records
)

print("Task trovati nel JSONL:")

for task_name, count in sorted(
    raw_task_counts.items(),
    key=lambda item: str(item[0]),
):
    print(
        f"- {task_name!r}: {count}"
    )


forbidden_tasks_found = (
    set(raw_task_counts)
    & FORBIDDEN_TASKS
)

if forbidden_tasks_found:
    raise RuntimeError(
        "Il JSONL contiene task che devono essere eliminati: "
        + ", ".join(
            sorted(forbidden_tasks_found)
        )
    )


unknown_tasks = (
    set(raw_task_counts)
    - set(TASK_CONFIG)
)

if unknown_tasks:
    raise RuntimeError(
        "Il JSONL contiene task non configurati: "
        + ", ".join(
            str(task)
            for task in sorted(
                unknown_tasks,
                key=str,
            )
        )
    )


missing_tasks = (
    set(TASK_CONFIG)
    - set(raw_task_counts)
)

if missing_tasks:
    raise RuntimeError(
        "Nel JSONL mancano i seguenti task: "
        + ", ".join(
            sorted(missing_tasks)
        )
    )

print("Controllo preliminare dei task superato.")

Task trovati nel JSONL:
- 'bias_detection': 20
- 'grounded_answer_generation': 20
- 'mnli_style': 20
- 'paraphrase_style': 20
- 'qqp_style': 20
- 'rte_style': 20
- 'sst2_style': 20
- 'stsb_style': 20
Controllo preliminare dei task superato.


In [9]:
count_errors = []

for task_name, expected_count in EXPECTED_TASK_COUNTS.items():
    actual_count = raw_task_counts.get(
        task_name,
        0,
    )

    if actual_count != expected_count:
        count_errors.append(
            f"{task_name}: "
            f"trovati {actual_count}, "
            f"attesi {expected_count}"
        )


if len(raw_records) != EXPECTED_TOTAL_RECORDS:
    count_errors.append(
        f"totale: trovati {len(raw_records)}, "
        f"attesi {EXPECTED_TOTAL_RECORDS}"
    )


if count_errors:
    raise RuntimeError(
        "Conteggi del test set non validi:\n- "
        + "\n- ".join(count_errors)
    )


print("Conteggi corretti.")
print("Totale:", len(raw_records))

for task_name in TASK_CONFIG:
    print(
        f"- {task_name}: "
        f"{raw_task_counts[task_name]}"
    )

Conteggi corretti.
Totale: 160
- sst2_style: 20
- paraphrase_style: 20
- stsb_style: 20
- qqp_style: 20
- mnli_style: 20
- rte_style: 20
- bias_detection: 20
- grounded_answer_generation: 20


In [10]:
example_ids = []

missing_id_records = []

for record in raw_records:
    example_id = record.get(
        "example_id"
    )

    if example_id is None:
        missing_id_records.append(
            record.get("_source_line")
        )

        continue

    example_id = str(
        example_id
    ).strip()

    if not example_id:
        missing_id_records.append(
            record.get("_source_line")
        )

        continue

    example_ids.append(
        example_id
    )


if missing_id_records:
    raise RuntimeError(
        "Sono presenti record senza example_id "
        "alle righe: "
        + ", ".join(
            str(line)
            for line in missing_id_records
        )
    )


duplicate_ids = [
    example_id
    for example_id, count
    in Counter(example_ids).items()
    if count > 1
]

if duplicate_ids:
    raise RuntimeError(
        "example_id duplicati: "
        + ", ".join(
            sorted(duplicate_ids)
        )
    )


if "GEN_001" not in example_ids:
    raise RuntimeError(
        "Il record GEN_001 non è presente. "
        "Aggiungere il record generativo mancante "
        "prima di proseguire."
    )


expected_generation_ids = {
    f"GEN_{index:03d}"
    for index in range(1, 21)
}

actual_generation_ids = {
    str(record["example_id"]).strip()
    for record in raw_records
    if record.get("task_name")
    == "grounded_answer_generation"
}

missing_generation_ids = (
    expected_generation_ids
    - actual_generation_ids
)

unexpected_generation_ids = (
    actual_generation_ids
    - expected_generation_ids
)

if missing_generation_ids:
    raise RuntimeError(
        "Mancano i seguenti identificativi generativi: "
        + ", ".join(
            sorted(missing_generation_ids)
        )
    )

if unexpected_generation_ids:
    raise RuntimeError(
        "Sono presenti identificativi generativi inattesi: "
        + ", ".join(
            sorted(unexpected_generation_ids)
        )
    )


print("Controllo identificativi superato.")
print("example_id univoci:", len(example_ids))
print("Record generativi:", len(actual_generation_ids))

Controllo identificativi superato.
example_id univoci: 160
Record generativi: 20


In [11]:
COMMON_FIELDS = {
    "example_id",
    "task_name",
    "task_type",
    "y_true",
    "gold",
    "gold_label",
    "gold_score",
    "allowed_labels",
    "inputs",
    "metadata",
    "_source_line",
}


def resolve_gold_value(
    record: dict,
):
    """
    Recupera la ground truth utilizzando diversi nomi
    eventualmente presenti nel dataset.
    """

    accepted_fields = [
        "y_true",
        "gold",
        "gold_label",
        "gold_score",
    ]

    for field_name in accepted_fields:
        if (
            field_name in record
            and record[field_name] is not None
        ):
            return record[field_name]

    raise ValueError(
        "Ground truth mancante. "
        "Campi supportati: "
        + ", ".join(accepted_fields)
    )


def normalize_string_label(
    value,
) -> str:
    """
    Normalizza una label classificatoria come stringa.
    """

    if value is None:
        raise ValueError(
            "La ground truth non può essere None."
        )

    normalized_value = str(
        value
    ).strip()

    if not normalized_value:
        raise ValueError(
            "La ground truth non può essere vuota."
        )

    return normalized_value


def normalize_stsb_score(
    value,
) -> int:
    """
    Converte e valida un punteggio STS-B intero tra 0 e 5.
    """

    if isinstance(value, bool):
        raise ValueError(
            "Un valore booleano non è un punteggio STS-B valido."
        )

    try:
        numeric_value = float(value)

    except (TypeError, ValueError) as error:
        raise ValueError(
            f"Punteggio STS-B non numerico: {value!r}"
        ) from error

    if not numeric_value.is_integer():
        raise ValueError(
            "Il punteggio STS-B deve essere intero. "
            f"Valore ricevuto: {numeric_value}"
        )

    integer_value = int(
        numeric_value
    )

    if integer_value not in {
        0,
        1,
        2,
        3,
        4,
        5,
    }:
        raise ValueError(
            "Il punteggio STS-B deve essere "
            f"compreso tra 0 e 5: {integer_value}"
        )

    return integer_value

In [12]:
def normalize_allowed_labels(
    record: dict,
    task_name: str,
):
    """
    Normalizza allowed_labels.

    Per i task classificatori e STS-B deve coincidere
    con la configurazione ufficiale.

    Per il task generativo viene restituito None.
    """

    expected_labels = TASK_CONFIG[
        task_name
    ]["allowed_labels"]

    if expected_labels is None:
        return None

    raw_labels = record.get(
        "allowed_labels"
    )

    if raw_labels is None:
        raise ValueError(
            f"allowed_labels mancante per il task {task_name}"
        )

    if isinstance(raw_labels, list):
        normalized_labels = raw_labels

    elif isinstance(raw_labels, tuple):
        normalized_labels = list(
            raw_labels
        )

    elif isinstance(raw_labels, str):
        normalized_labels = [
            item.strip()
            for item in raw_labels.split(",")
            if item.strip()
        ]

    else:
        raise TypeError(
            "allowed_labels deve essere una lista "
            "oppure una stringa separata da virgole. "
            f"Tipo ricevuto: {type(raw_labels).__name__}"
        )

    if task_name == "stsb_style":
        normalized_labels = [
            normalize_stsb_score(label)
            for label in normalized_labels
        ]

    else:
        normalized_labels = [
            normalize_string_label(label)
            for label in normalized_labels
        ]

    return normalized_labels

In [13]:
def extract_inputs(
    record: dict,
    task_name: str,
) -> dict:
    """
    Estrae i campi che verranno mostrati al modello.
    """

    expected_input_fields = TASK_CONFIG[
        task_name
    ]["input_fields"]

    nested_inputs = record.get(
        "inputs"
    )

    if nested_inputs is None:
        nested_inputs = {}

    if not isinstance(
        nested_inputs,
        dict,
    ):
        raise TypeError(
            "Il campo inputs deve essere un oggetto JSON."
        )

    normalized_inputs = {}

    for field_name in expected_input_fields:
        value = nested_inputs.get(
            field_name
        )

        if value is None:
            value = record.get(
                field_name
            )

        if value is None:
            raise ValueError(
                f"Campo input mancante: {field_name!r}"
            )

        value = str(
            value
        ).strip()

        if not value:
            raise ValueError(
                f"Campo input vuoto: {field_name!r}"
            )

        normalized_inputs[
            field_name
        ] = value

    return normalized_inputs

In [14]:
def extract_metadata(
    record: dict,
    task_name: str,
) -> dict:
    """
    Conserva eventuali campi aggiuntivi senza inserirli nel prompt.
    """

    metadata = {}

    existing_metadata = record.get(
        "metadata"
    )

    if existing_metadata is not None:
        if not isinstance(
            existing_metadata,
            dict,
        ):
            raise TypeError(
                "Il campo metadata deve essere un oggetto JSON."
            )

        metadata.update(
            existing_metadata
        )

    input_fields = set(
        TASK_CONFIG[
            task_name
        ]["input_fields"]
    )

    excluded_fields = (
        COMMON_FIELDS
        | input_fields
    )

    for key, value in record.items():
        if key in excluded_fields:
            continue

        metadata[key] = value

    if record.get(
        "_source_line"
    ) is not None:
        metadata.setdefault(
            "source_line",
            record["_source_line"],
        )

    return metadata

In [15]:
def normalize_record(
    record: dict,
) -> dict:
    """
    Converte un record grezzo nello schema comune di valutazione.
    """

    task_name = record.get(
        "task_name"
    )

    if task_name is None:
        raise ValueError(
            "task_name mancante."
        )

    task_name = str(
        task_name
    ).strip()

    if task_name in FORBIDDEN_TASKS:
        raise ValueError(
            f"Il task {task_name!r} è escluso dal benchmark."
        )

    if task_name not in TASK_CONFIG:
        raise ValueError(
            f"Task non configurato: {task_name!r}"
        )

    example_id = record.get(
        "example_id"
    )

    if example_id is None:
        raise ValueError(
            "example_id mancante."
        )

    example_id = str(
        example_id
    ).strip()

    if not example_id:
        raise ValueError(
            "example_id vuoto."
        )

    config = TASK_CONFIG[
        task_name
    ]

    raw_gold_value = resolve_gold_value(
        record
    )

    if task_name == "stsb_style":
        y_true = normalize_stsb_score(
            raw_gold_value
        )

    else:
        y_true = normalize_string_label(
            raw_gold_value
        )

    allowed_labels = normalize_allowed_labels(
        record,
        task_name,
    )

    inputs = extract_inputs(
        record,
        task_name,
    )

    metadata = extract_metadata(
        record,
        task_name,
    )

    return {
        "example_id": example_id,
        "task_name": task_name,
        "task_type": config["task_type"],
        "y_true": y_true,
        "allowed_labels": allowed_labels,
        "inputs": inputs,
        "metadata": metadata,
    }

In [16]:
normalized_records = []

normalization_errors = []

for record_index, raw_record in enumerate(
    raw_records,
    start=1,
):
    try:
        normalized_record = normalize_record(
            raw_record
        )

        normalized_records.append(
            normalized_record
        )

    except Exception as error:
        normalization_errors.append({
            "record_index": record_index,
            "source_line": raw_record.get(
                "_source_line"
            ),
            "example_id": raw_record.get(
                "example_id"
            ),
            "task_name": raw_record.get(
                "task_name"
            ),
            "error_type": type(
                error
            ).__name__,
            "error": str(error),
        })


if normalization_errors:
    print(
        json.dumps(
            normalization_errors[:50],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Normalizzazione fallita per "
        f"{len(normalization_errors)} record."
    )


print(
    "Record normalizzati:",
    len(normalized_records),
)

Record normalizzati: 160


In [17]:
def validate_normalized_dataset(
    records: list[dict],
) -> None:
    errors = []

    if len(records) != EXPECTED_TOTAL_RECORDS:
        errors.append(
            f"Numero complessivo di record non valido: "
            f"{len(records)} invece di {EXPECTED_TOTAL_RECORDS}."
        )

    normalized_ids = [
        record["example_id"]
        for record in records
    ]

    duplicate_ids = [
        example_id
        for example_id, count
        in Counter(normalized_ids).items()
        if count > 1
    ]

    if duplicate_ids:
        errors.append(
            "example_id duplicati: "
            + ", ".join(
                sorted(duplicate_ids)
            )
        )

    normalized_task_counts = Counter(
        record["task_name"]
        for record in records
    )

    for task_name, expected_count in EXPECTED_TASK_COUNTS.items():
        actual_count = normalized_task_counts.get(
            task_name,
            0,
        )

        if actual_count != expected_count:
            errors.append(
                f"{task_name}: "
                f"{actual_count} record invece di {expected_count}."
            )

    for record in records:
        example_id = record[
            "example_id"
        ]

        task_name = record[
            "task_name"
        ]

        y_true = record[
            "y_true"
        ]

        allowed_labels = record[
            "allowed_labels"
        ]

        expected_labels = TASK_CONFIG[
            task_name
        ]["allowed_labels"]

        if allowed_labels != expected_labels:
            errors.append(
                f"{example_id}: allowed_labels non coerenti. "
                f"Trovate={allowed_labels!r}; "
                f"attese={expected_labels!r}."
            )

        if allowed_labels is not None:
            if y_true not in allowed_labels:
                errors.append(
                    f"{example_id}: y_true={y_true!r} "
                    f"non presente in allowed_labels="
                    f"{allowed_labels!r}."
                )

        else:
            if task_name != "grounded_answer_generation":
                errors.append(
                    f"{example_id}: allowed_labels assenti "
                    f"per il task {task_name}."
                )

    if errors:
        print(
            "\n".join(
                f"- {error}"
                for error in errors[:100]
            )
        )

        raise RuntimeError(
            f"Validazione fallita: {len(errors)} errori."
        )


validate_normalized_dataset(
    normalized_records
)

normalized_task_counts = Counter(
    record["task_name"]
    for record in normalized_records
)

print("Validazione strutturale superata.")

for task_name in TASK_CONFIG:
    print(
        f"- {task_name}: "
        f"{normalized_task_counts[task_name]}"
    )

Validazione strutturale superata.
- sst2_style: 20
- paraphrase_style: 20
- stsb_style: 20
- qqp_style: 20
- mnli_style: 20
- rte_style: 20
- bias_detection: 20
- grounded_answer_generation: 20


In [18]:
label_distribution = defaultdict(
    Counter
)

for record in normalized_records:
    label_distribution[
        record["task_name"]
    ][
        record["y_true"]
    ] += 1


for task_name in TASK_CONFIG:
    print("\n" + "=" * 80)
    print("TASK:", task_name)

    for label, count in sorted(
        label_distribution[task_name].items(),
        key=lambda item: str(item[0]),
    ):
        print(
            f"- {label!r}: {count}"
        )


TASK: sst2_style
- 'compliant': 10
- 'non_compliant': 10

TASK: paraphrase_style
- 'equivalent': 11
- 'not_equivalent': 9

TASK: stsb_style
- 0: 3
- 1: 3
- 2: 3
- 3: 4
- 4: 4
- 5: 3

TASK: qqp_style
- 'duplicate': 12
- 'not_duplicate': 8

TASK: mnli_style
- 'contradiction': 7
- 'entailment': 8
- 'neutral': 5

TASK: rte_style
- 'entailment': 10
- 'not_entailment': 10

TASK: bias_detection
- 'biased': 10
- 'unbiased': 10

TASK: grounded_answer_generation
- 'A multi-disciplinary approach ensures comprehensive protection by involving HR, legal experts, victim support services, and, where necessary, the criminal justice system.': 1
- 'Fixed gender expectations restrict individual freedom, limit professional aspirations, and act as a root cause of inequality and discrimination in career progression.': 1
- 'HR must comply with the request by providing the average pay levels, broken down by sex, for categories of workers performing the same work or work of equal value.': 1
- 'Management shoul

In [19]:
if OFFLINE_MODE:
    os.environ[
        "HF_HUB_OFFLINE"
    ] = "1"

    os.environ[
        "TRANSFORMERS_OFFLINE"
    ] = "1"


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    local_files_only=OFFLINE_MODE,
)


if tokenizer.pad_token_id is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )


if tokenizer.chat_template is None:
    raise RuntimeError(
        "Il tokenizer non contiene un chat template. "
        "Non è possibile costruire in modo affidabile "
        "i prompt per Qwen."
    )


print("Tokenizer caricato:", MODEL_ID)
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("Chat template disponibile:", tokenizer.chat_template is not None)

Tokenizer caricato: Qwen/Qwen2.5-3B-Instruct
Pad token: <|endoftext|>
EOS token: <|im_end|>
Chat template disponibile: True


In [20]:
def labels_as_text(
    labels: list,
) -> str:
    return ", ".join(
        str(label)
        for label in labels
    )

In [21]:
def build_messages(
    record: dict,
) -> list[dict]:
    """
    Costruisce system message e user message
    per ciascun task.
    """

    task_name = record[
        "task_name"
    ]

    inputs = record[
        "inputs"
    ]

    allowed_labels = record[
        "allowed_labels"
    ]

    if task_name == "sst2_style":
        system_content = (
            "You are evaluating regulatory compliance. "
            "Classify the scenario using exactly one of the "
            "allowed labels. Return only the label, without "
            "explanations, punctuation, or additional text."
        )

        user_content = (
            "Determine whether the following scenario is "
            "compliant or non-compliant.\n\n"
            f"Scenario:\n{inputs['text']}\n\n"
            f"Allowed labels: "
            f"{labels_as_text(allowed_labels)}"
        )

    elif task_name == "paraphrase_style":
        system_content = (
            "Determine whether the two statements express the "
            "same substantive meaning. Use exactly one of the "
            "allowed labels. Return only the label, without "
            "explanations, punctuation, or additional text."
        )

        user_content = (
            "Compare the following statements.\n\n"
            f"Statement 1:\n{inputs['sentence_1']}\n\n"
            f"Statement 2:\n{inputs['sentence_2']}\n\n"
            f"Allowed labels: "
            f"{labels_as_text(allowed_labels)}"
        )

    elif task_name == "stsb_style":
        system_content = (
            "Evaluate the semantic similarity between two "
            "statements using an integer score from 0 to 5. "
            "Use 0 for completely unrelated meanings and 5 "
            "for semantically equivalent meanings. Return only "
            "one integer: 0, 1, 2, 3, 4, or 5. Do not return "
            "decimals, explanations, punctuation, or additional text."
        )

        user_content = (
            "Assign an integer semantic similarity score.\n\n"
            f"Statement 1:\n{inputs['sentence_1']}\n\n"
            f"Statement 2:\n{inputs['sentence_2']}\n\n"
            "Allowed integer scores: 0, 1, 2, 3, 4, 5"
        )

    elif task_name == "qqp_style":
        system_content = (
            "Determine whether the two questions request the "
            "same information. Use exactly one of the allowed "
            "labels. Return only the label, without explanations, "
            "punctuation, or additional text."
        )

        user_content = (
            "Compare the following questions.\n\n"
            f"Question 1:\n{inputs['question_1']}\n\n"
            f"Question 2:\n{inputs['question_2']}\n\n"
            f"Allowed labels: "
            f"{labels_as_text(allowed_labels)}"
        )

    elif task_name == "mnli_style":
        system_content = (
            "Classify the logical relationship between a premise "
            "and a hypothesis. Use entailment when the hypothesis "
            "follows from the premise, contradiction when the "
            "hypothesis conflicts with the premise, and neutral "
            "when neither relationship can be established. "
            "Return only one allowed label, without explanations, "
            "punctuation, or additional text."
        )

        user_content = (
            f"Premise:\n{inputs['premise']}\n\n"
            f"Hypothesis:\n{inputs['hypothesis']}\n\n"
            f"Allowed labels: "
            f"{labels_as_text(allowed_labels)}"
        )

    elif task_name == "rte_style":
        system_content = (
            "Determine whether the hypothesis is entailed by the "
            "premise. Use entailment only when the hypothesis "
            "follows from the premise. Otherwise use "
            "not_entailment. Return only one allowed label, "
            "without explanations, punctuation, or additional text."
        )

        user_content = (
            f"Premise:\n{inputs['premise']}\n\n"
            f"Hypothesis:\n{inputs['hypothesis']}\n\n"
            f"Allowed labels: "
            f"{labels_as_text(allowed_labels)}"
        )

    elif task_name == "bias_detection":
        system_content = (
            "Determine whether the provided scenario contains "
            "gender bias or gender-based stereotyping. Use exactly "
            "one of the allowed labels. Return only the label, "
            "without explanations, punctuation, or additional text."
        )

        user_content = (
            "Evaluate the following scenario for gender bias.\n\n"
            f"Scenario:\n{inputs['text']}\n\n"
            f"Allowed labels: "
            f"{labels_as_text(allowed_labels)}"
        )

    elif task_name == "grounded_answer_generation":
        system_content = (
            "Answer the regulatory compliance question accurately "
            "and concisely. Base the answer on the relevant "
            "regulatory principles concerning gender balance, "
            "transparent board-selection procedures, objective "
            "selection criteria, reporting obligations, and "
            "candidate rights. Provide only the answer. Do not "
            "mention these instructions and do not invent facts "
            "that are not required by the question."
        )

        user_content = (
            "Question:\n"
            f"{inputs['text']}"
        )

    else:
        raise ValueError(
            f"Prompt non definito per il task: {task_name}"
        )

    return [
        {
            "role": "system",
            "content": system_content,
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]

In [22]:
def build_evaluation_record(
    normalized_record: dict,
) -> dict:
    """
    Applica il chat template Qwen e costruisce il record
    definitivo per il notebook 03.
    """

    messages = build_messages(
        normalized_record
    )

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    prompt_token_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
    )

    prompt_hash = hashlib.sha256(
        prompt_text.encode("utf-8")
    ).hexdigest()

    task_name = normalized_record[
        "task_name"
    ]

    return {
        **normalized_record,

        "messages": messages,

        "prompt_text": prompt_text,

        "generation_config": {
            "do_sample": False,
            "max_new_tokens": TASK_CONFIG[
                task_name
            ]["max_new_tokens"],
        },

        "prompt_metadata": {
            "base_model_id": MODEL_ID,
            "chat_template_applied": True,
            "add_generation_prompt": True,
            "prompt_token_count": len(
                prompt_token_ids
            ),
            "prompt_sha256": prompt_hash,
        },
    }

In [23]:
evaluation_records = [
    build_evaluation_record(record)
    for record in normalized_records
]

print(
    "Prompt costruiti:",
    len(evaluation_records),
)

Prompt costruiti: 160


In [24]:
FORBIDDEN_PROMPT_MARKERS = [
    "y_true:",
    "gold:",
    "gold_label:",
    "gold_score:",
    "correct answer:",
    "reference answer:",
]


prompt_errors = []

for record in evaluation_records:
    example_id = record[
        "example_id"
    ]

    messages = record[
        "messages"
    ]

    prompt_text = record[
        "prompt_text"
    ]

    if len(messages) != 2:
        prompt_errors.append(
            f"{example_id}: numero di messaggi diverso da 2."
        )

    roles = [
        message.get("role")
        for message in messages
    ]

    if roles != [
        "system",
        "user",
    ]:
        prompt_errors.append(
            f"{example_id}: ruoli non validi: {roles!r}."
        )

    if any(
        message.get("role") == "assistant"
        for message in messages
    ):
        prompt_errors.append(
            f"{example_id}: il prompt contiene già "
            "un messaggio assistant."
        )

    if not prompt_text.strip():
        prompt_errors.append(
            f"{example_id}: prompt vuoto."
        )

    prompt_text_lower = (
        prompt_text.lower()
    )

    found_markers = [
        marker
        for marker in FORBIDDEN_PROMPT_MARKERS
        if marker in prompt_text_lower
    ]

    if found_markers:
        prompt_errors.append(
            f"{example_id}: possibili marker di leakage: "
            f"{found_markers!r}."
        )


if prompt_errors:
    print(
        "\n".join(
            f"- {error}"
            for error in prompt_errors[:100]
        )
    )

    raise RuntimeError(
        f"Controllo dei prompt fallito: "
        f"{len(prompt_errors)} errori."
    )


print("Controllo dei prompt superato.")

Controllo dei prompt superato.


In [25]:
prompt_statistics = {}

for task_name in TASK_CONFIG:
    task_records = [
        record
        for record in evaluation_records
        if record["task_name"] == task_name
    ]

    token_counts = [
        record[
            "prompt_metadata"
        ][
            "prompt_token_count"
        ]
        for record in task_records
    ]

    prompt_statistics[
        task_name
    ] = {
        "examples": len(
            task_records
        ),
        "minimum_prompt_tokens": min(
            token_counts
        ),
        "maximum_prompt_tokens": max(
            token_counts
        ),
        "average_prompt_tokens": round(
            sum(token_counts)
            / len(token_counts),
            2,
        ),
    }


print(
    json.dumps(
        prompt_statistics,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "sst2_style": {
    "examples": 20,
    "minimum_prompt_tokens": 2,
    "maximum_prompt_tokens": 2,
    "average_prompt_tokens": 2.0
  },
  "paraphrase_style": {
    "examples": 20,
    "minimum_prompt_tokens": 2,
    "maximum_prompt_tokens": 2,
    "average_prompt_tokens": 2.0
  },
  "stsb_style": {
    "examples": 20,
    "minimum_prompt_tokens": 2,
    "maximum_prompt_tokens": 2,
    "average_prompt_tokens": 2.0
  },
  "qqp_style": {
    "examples": 20,
    "minimum_prompt_tokens": 2,
    "maximum_prompt_tokens": 2,
    "average_prompt_tokens": 2.0
  },
  "mnli_style": {
    "examples": 20,
    "minimum_prompt_tokens": 2,
    "maximum_prompt_tokens": 2,
    "average_prompt_tokens": 2.0
  },
  "rte_style": {
    "examples": 20,
    "minimum_prompt_tokens": 2,
    "maximum_prompt_tokens": 2,
    "average_prompt_tokens": 2.0
  },
  "bias_detection": {
    "examples": 20,
    "minimum_prompt_tokens": 2,
    "maximum_prompt_tokens": 2,
    "average_prompt_tokens": 2.0
  },
  "grounde

In [26]:
for task_name in TASK_CONFIG:
    task_records = [
        record
        for record in evaluation_records
        if record["task_name"] == task_name
    ]

    example = task_records[0]

    print("\n" + "=" * 100)
    print("TASK:", task_name)
    print("EXAMPLE ID:", example["example_id"])
    print("Y TRUE:", example["y_true"])
    print(
        "ALLOWED LABELS:",
        example["allowed_labels"],
    )
    print(
        "PROMPT TOKENS:",
        example[
            "prompt_metadata"
        ][
            "prompt_token_count"
        ],
    )
    print("-" * 100)
    print(example["prompt_text"])


TASK: sst2_style
EXAMPLE ID: SST2_001
Y TRUE: compliant
ALLOWED LABELS: ['compliant', 'non_compliant']
PROMPT TOKENS: 2
----------------------------------------------------------------------------------------------------
<|im_start|>system
You are evaluating regulatory compliance. Classify the scenario using exactly one of the allowed labels. Return only the label, without explanations, punctuation, or additional text.<|im_end|>
<|im_start|>user
Determine whether the following scenario is compliant or non-compliant.

Scenario:
A listed company's nomination committee publishes a detailed skills matrix before evaluating candidates for non-executive roles.

Allowed labels: compliant, non_compliant<|im_end|>
<|im_start|>assistant


TASK: paraphrase_style
EXAMPLE ID: MRPC_001
Y TRUE: equivalent
ALLOWED LABELS: ['equivalent', 'not_equivalent']
PROMPT TOKENS: 2
----------------------------------------------------------------------------------------------------
<|im_start|>system
Determine wh

In [27]:
gen_001_records = [
    record
    for record in evaluation_records
    if record["example_id"] == "GEN_001"
]

if len(gen_001_records) != 1:
    raise RuntimeError(
        "GEN_001 deve comparire esattamente una volta."
    )

gen_001 = gen_001_records[0]

print(
    json.dumps(
        gen_001,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "example_id": "GEN_001",
  "task_name": "grounded_answer_generation",
  "task_type": "generation",
  "y_true": "The entity must pursue the objective of achieving at least 40% representation of the underrepresented sex among non-executive directors, or 33% among all directors, by implementing transparent selection processes.",
  "allowed_labels": null,
  "inputs": {
    "text": "A listed corporate entity currently has only 20% female representation among non-executive directors. What objective must it pursue?"
  },
  "metadata": {
    "source_line": 151
  },
  "messages": [
    {
      "role": "system",
      "content": "Answer the regulatory compliance question accurately and concisely. Base the answer on the relevant regulatory principles concerning gender balance, transparent board-selection procedures, objective selection criteria, reporting obligations, and candidate rights. Provide only the answer. Do not mention these instructions and do not invent facts that are not required

In [28]:
write_jsonl(
    OUTPUT_NORMALIZED_PATH,
    normalized_records,
)

print(
    "Dataset normalizzato salvato in:",
    OUTPUT_NORMALIZED_PATH,
)

Dataset normalizzato salvato in: C:\Users\fede2\Desktop\Project\data\processed\evaluation\evaluation_normalized.jsonl


In [29]:
write_jsonl(
    OUTPUT_ALL_PROMPTS_PATH,
    evaluation_records,
)

print(
    "Dataset completo dei prompt salvato in:",
    OUTPUT_ALL_PROMPTS_PATH,
)

Dataset completo dei prompt salvato in: C:\Users\fede2\Desktop\Project\data\processed\evaluation\evaluation_prompts_all.jsonl


In [30]:
task_output_paths = {}

for task_name in TASK_CONFIG:
    task_records = [
        record
        for record in evaluation_records
        if record["task_name"] == task_name
    ]

    task_output_path = (
        OUTPUT_BY_TASK_DIR
        / f"{task_name}_evaluation_prompts.jsonl"
    )

    write_jsonl(
        task_output_path,
        task_records,
    )

    task_output_paths[
        task_name
    ] = task_output_path


print("File per task salvati:")

for task_name, path in task_output_paths.items():
    print(
        f"- {task_name}: {path}"
    )

File per task salvati:
- sst2_style: C:\Users\fede2\Desktop\Project\data\processed\evaluation\by_task\sst2_style_evaluation_prompts.jsonl
- paraphrase_style: C:\Users\fede2\Desktop\Project\data\processed\evaluation\by_task\paraphrase_style_evaluation_prompts.jsonl
- stsb_style: C:\Users\fede2\Desktop\Project\data\processed\evaluation\by_task\stsb_style_evaluation_prompts.jsonl
- qqp_style: C:\Users\fede2\Desktop\Project\data\processed\evaluation\by_task\qqp_style_evaluation_prompts.jsonl
- mnli_style: C:\Users\fede2\Desktop\Project\data\processed\evaluation\by_task\mnli_style_evaluation_prompts.jsonl
- rte_style: C:\Users\fede2\Desktop\Project\data\processed\evaluation\by_task\rte_style_evaluation_prompts.jsonl
- bias_detection: C:\Users\fede2\Desktop\Project\data\processed\evaluation\by_task\bias_detection_evaluation_prompts.jsonl
- grounded_answer_generation: C:\Users\fede2\Desktop\Project\data\processed\evaluation\by_task\grounded_answer_generation_evaluation_prompts.jsonl


In [31]:
saved_all_records = read_jsonl(
    OUTPUT_ALL_PROMPTS_PATH
)

if len(saved_all_records) != EXPECTED_TOTAL_RECORDS:
    raise RuntimeError(
        "Il file salvato contiene un numero errato di record: "
        f"{len(saved_all_records)} invece di "
        f"{EXPECTED_TOTAL_RECORDS}."
    )


saved_ids = {
    record["example_id"]
    for record in saved_all_records
}

expected_ids = {
    record["example_id"]
    for record in evaluation_records
}

if saved_ids != expected_ids:
    missing_ids = (
        expected_ids
        - saved_ids
    )

    unexpected_ids = (
        saved_ids
        - expected_ids
    )

    raise RuntimeError(
        "Gli identificativi salvati non coincidono.\n"
        f"Mancanti: {sorted(missing_ids)}\n"
        f"Inattesi: {sorted(unexpected_ids)}"
    )


for task_name, output_path in task_output_paths.items():
    saved_task_records = read_jsonl(
        output_path
    )

    expected_count = EXPECTED_TASK_COUNTS[
        task_name
    ]

    if len(saved_task_records) != expected_count:
        raise RuntimeError(
            f"File {output_path.name}: "
            f"{len(saved_task_records)} record "
            f"invece di {expected_count}."
        )


print("Verifica dei file salvati superata.")
print("Record complessivi verificati:", len(saved_all_records))
print("example_id verificati:", len(saved_ids))

Verifica dei file salvati superata.
Record complessivi verificati: 160
example_id verificati: 160


In [32]:
saved_all_records = read_jsonl(
    OUTPUT_ALL_PROMPTS_PATH
)

if len(saved_all_records) != EXPECTED_TOTAL_RECORDS:
    raise RuntimeError(
        "Il file salvato contiene un numero errato di record: "
        f"{len(saved_all_records)} invece di "
        f"{EXPECTED_TOTAL_RECORDS}."
    )


saved_ids = {
    record["example_id"]
    for record in saved_all_records
}

expected_ids = {
    record["example_id"]
    for record in evaluation_records
}

if saved_ids != expected_ids:
    missing_ids = (
        expected_ids
        - saved_ids
    )

    unexpected_ids = (
        saved_ids
        - expected_ids
    )

    raise RuntimeError(
        "Gli identificativi salvati non coincidono.\n"
        f"Mancanti: {sorted(missing_ids)}\n"
        f"Inattesi: {sorted(unexpected_ids)}"
    )


for task_name, output_path in task_output_paths.items():
    saved_task_records = read_jsonl(
        output_path
    )

    expected_count = EXPECTED_TASK_COUNTS[
        task_name
    ]

    if len(saved_task_records) != expected_count:
        raise RuntimeError(
            f"File {output_path.name}: "
            f"{len(saved_task_records)} record "
            f"invece di {expected_count}."
        )


print("Verifica dei file salvati superata.")
print("Record complessivi verificati:", len(saved_all_records))
print("example_id verificati:", len(saved_ids))

Verifica dei file salvati superata.
Record complessivi verificati: 160
example_id verificati: 160


In [33]:
manifest = {
    "notebook": "02_build_evaluation_prompts.ipynb",

    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),

    "project_root": str(
        PROJECT_ROOT
    ),

    "input": {
        "path": str(
            INPUT_JSONL_PATH
        ),
        "records": len(
            raw_records
        ),
    },

    "benchmark": {
        "included_tasks": list(
            TASK_CONFIG.keys()
        ),
        "excluded_tasks": sorted(
            FORBIDDEN_TASKS
        ),
        "expected_total_records": EXPECTED_TOTAL_RECORDS,
        "actual_total_records": len(
            evaluation_records
        ),
        "expected_task_counts": EXPECTED_TASK_COUNTS,
        "actual_task_counts": dict(
            normalized_task_counts
        ),
        "generation_record_gen_001_present": (
            "GEN_001" in saved_ids
        ),
    },

    "stsb": {
        "score_type": "integer",
        "allowed_scores": [
            0,
            1,
            2,
            3,
            4,
            5,
        ],
        "decimal_scores_allowed": False,
    },

    "tokenizer": {
        "model_id": MODEL_ID,
        "offline_mode": OFFLINE_MODE,
        "chat_template_available": (
            tokenizer.chat_template
            is not None
        ),
        "pad_token": tokenizer.pad_token,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token": tokenizer.eos_token,
        "eos_token_id": tokenizer.eos_token_id,
    },

    "outputs": {
        "normalized_dataset": str(
            OUTPUT_NORMALIZED_PATH
        ),

        "all_prompts": str(
            OUTPUT_ALL_PROMPTS_PATH
        ),

        "by_task": {
            task_name: str(path)
            for task_name, path
            in task_output_paths.items()
        },
    },

    "prompt_statistics": prompt_statistics,

    "label_distribution": {
        task_name: {
            str(label): count
            for label, count
            in distribution.items()
        }
        for task_name, distribution
        in label_distribution.items()
    },
}


write_json(
    MANIFEST_PATH,
    manifest,
)

print(
    "Manifest salvato in:",
    MANIFEST_PATH,
)

Manifest salvato in: C:\Users\fede2\Desktop\Project\manifests\02_build_evaluation_prompts_manifest.json


In [34]:
print("\n" + "=" * 100)
print("NOTEBOOK 02 COMPLETATO")
print("=" * 100)

print("\nInput:")
print(INPUT_JSONL_PATH)

print("\nRecord elaborati:")
print(len(evaluation_records))

print("\nTask inclusi:")

for task_name in TASK_CONFIG:
    print(
        f"- {task_name}: "
        f"{normalized_task_counts[task_name]}"
    )

print("\nTask esclusi:")
for task_name in sorted(FORBIDDEN_TASKS):
    print(
        f"- {task_name}"
    )

print("\nConfigurazione STS-B:")
print("- punteggi interi")
print("- valori ammessi: 0, 1, 2, 3, 4, 5")
print("- valori decimali non ammessi")

print("\nGEN_001 presente:")
print("GEN_001" in saved_ids)

print("\nFile pronto per il notebook 03:")
print(OUTPUT_ALL_PROMPTS_PATH)

print("\nManifest:")
print(MANIFEST_PATH)

print("\nNotebook completato correttamente.")


NOTEBOOK 02 COMPLETATO

Input:
C:\Users\fede2\Desktop\Project\data\raw\evaluation\test_set.jsonl

Record elaborati:
160

Task inclusi:
- sst2_style: 20
- paraphrase_style: 20
- stsb_style: 20
- qqp_style: 20
- mnli_style: 20
- rte_style: 20
- bias_detection: 20
- grounded_answer_generation: 20

Task esclusi:
- qnli_style

Configurazione STS-B:
- punteggi interi
- valori ammessi: 0, 1, 2, 3, 4, 5
- valori decimali non ammessi

GEN_001 presente:
True

File pronto per il notebook 03:
C:\Users\fede2\Desktop\Project\data\processed\evaluation\evaluation_prompts_all.jsonl

Manifest:
C:\Users\fede2\Desktop\Project\manifests\02_build_evaluation_prompts_manifest.json

Notebook completato correttamente.
